**Eye-Tracking & Neurocognitive Intelligence | Reading Comprehension & Neurocognitive Modelling | Presented by: Reza Saadatyar (2026)**

**03 | Classical ML and Text-Based Baselines**

![Classical ML and Text-Based Baselines — scientific workflow](../../images/03_classical_ml_and_text_baselines.svg)

**Objective:** Compare gaze-only, text-only and combined classical predictors, including a training-fitted lexical TF-IDF comparator.

**Dataset:** Trial-level feature table, fixed participant assignments and local paragraph-text table from Notebook 02.

**Scope:** Fit fixed classical candidates and compare gaze, text and combined representations using Validation; save held-out scores.


**00. Colab Repository Setup**

In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/Eye-Tracking-Neurocognitive-Intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


**01. Imports and Research Environment**

In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("reading_comprehension_modelling")

# ---------------------------- 2. Standard Library ----------------------------
import subprocess
import sys

# --------------------------- 3. Study Dependencies ---------------------------
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(root / "reading_comprehension_modelling" / "requirements.txt")],
    check=True,
)

# ----------------- 4. Scientific Computing and Data Analysis -----------------
import pandas as pd

# ------------------- 5. Classical Modelling and Evaluation -------------------
from reading_models import classical_predictions, CLASSICAL
from reading_evaluation import metric

# ------------------- 6. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure

# ----------------- 7. Shared and Project Research Utilities ------------------
from core.output_download import download_outputs_from_colab
from onestop_io import validate_splits, validate_trial_index

# ------------------- 8. Output Directory and Setup Summary -------------------
out_dir = root / "reading_comprehension_modelling" / "output"
out_dir.mkdir(parents=True, exist_ok=True)
processed = root / "data" / "processed" / "onestop"
processed.mkdir(parents=True, exist_ok=True)
plot_style(
    font_family="DejaVu Sans", title_size=12.5,
    title_weight="normal", label_size=11.5,
    label_weight="normal", tick_size=11, legend_size=10,
    figure_dpi=100, figure_size=(9, 4),
)
print("Study outputs:", out_dir)
print("Local data directory:", processed)
print("Python:", sys.version.split()[0])


**02. Load Matched Trial Features and Participant Roles**

Read the fixed trial table and split assignments. The local paragraph table contains text for the lexical comparator; it must match the indexed trials. The model allowlists exclude all answer and identity fields.

In [ ]:
# =========== 02. Load Matched Trial Features and Participant Roles ===========

# ------------------- 1. Load and Reconcile Trial Features --------------------
identity = ["trial_id","participant_id","trial_index","article_batch",
            "article_key","paragraph_key","question_key","difficulty_level"]
table = pd.read_csv(out_dir/"02_trial_features.csv",dtype={c:str for c in identity})
index = pd.read_csv(out_dir/"01_trial_index.csv",dtype=str)
index["is_correct"] = index.is_correct.astype(int)
validate_trial_index(index,table)

# --------------- 2. Load Participant Roles and Paragraph Text ----------------
splits = pd.read_csv(out_dir/"01_participant_splits.csv",dtype=str)
paragraph_text = pd.read_csv(processed/"02_paragraph_text.csv",dtype=str)
validate_splits(table,splits)
print("Classical candidates:",CLASSICAL)


**03. Fit Classical Models and Lexical Baselines**

Compare a Training prevalence reference, Logistic Regression, Random Forest and XGBoost for gaze-only, text-only and combined scalar predictors. Also compare paragraph TF-IDF Logistic Regression with and without gaze features. Imputation, scaling, vocabulary and class weights are Training-only. The decision threshold is fixed at 0.5.

In [ ]:
# ============== 03. Fit Classical Models and Lexical Baselines ===============

# --------------------- 1. Fit Fixed Classical Candidates ---------------------
classical, importance = classical_predictions(table,splits,paragraph_text,seed=17)

# --------------------- 2. Summarise Prediction Coverage ----------------------
print("Prediction rows:",len(classical))


**04. Examine Validation Evidence**

Only Validation labels are scored here. Test scores are saved without outcome columns; Notebook 05 uses the fixed candidates and Validation selection before calculating Test metrics. Permutation importance for scalar predictors uses Validation only and measures model dependence, not causal effects.

In [ ]:
# ====================== 04. Examine Validation Evidence ======================

# -------------------- 1. Summarise Validation Performance --------------------
v = classical[classical.role.eq("validation")].merge(
    table[["trial_id","is_correct"]],on="trial_id",validate="many_to_one")
validation = pd.DataFrame([dict(protocol=p,model=m,**metric(g.is_correct,g.p_correct))
                           for (p,m),g in v.groupby(["protocol","model"])])
display(validation.sort_values(["protocol","macro_f1"],ascending=[True,False]))

# ----------------------- 2. Plot Validation Comparison -----------------------
fig, ax = plt.subplots(figsize=(11,6))
validation.pivot(index="model",columns="protocol",values="macro_f1").plot.barh(ax=ax)
ax.set(xlabel="Validation Macro F1",ylabel="",title="Classical and lexical candidates")
fig.tight_layout()
save_figure(fig,out_dir/"03_validation_comparison.png")
plt.show()


**05. Save Classical Prediction Scores**

Export one score per candidate and eligible Validation or Test trial. The scalar-model permutation summaries are also saved. No Test-informed fitting or selection occurs in this notebook.

In [ ]:
# =================== 05. Save Classical Prediction Scores ====================

# ---------------- 1. Save Predictions and Validation Evidence ----------------
classical.to_csv(out_dir/"03_classical_predictions.csv",index=False)
validation.to_csv(out_dir/"03_validation_metrics.csv",index=False)
importance.to_csv(out_dir/"03_validation_feature_importance.csv",index=False)

# ------------------------ 2. Package Current Outputs -------------------------
download_outputs_from_colab(out_dir,"03")
